# Modelos recurrentes 

Se comparan LSTM y GRU multi-salida y encoder-decoder con
covariables de calendario. Los modelos se entrenaron en Google Colab
con GPU Tesla T4, usando L=168 y H=24.

La evaluación de prueba utiliza el protocolo compartido:
92 orígenes diarios entre octubre y diciembre de 2014.

In [1]:
print("El kernel responde", flush=True)

El kernel responde


In [3]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import common as C

MODELOS = [
    "lstm_multi",
    "gru_multi",
    "seq2seq_lstm",
    "seq2seq_gru",
]

necesarios = [
    C.PANEL_PATH,
    *[C.PRED_DIR / f"{modelo}.parquet" for modelo in MODELOS],
]

faltantes = [str(p) for p in necesarios if not p.exists()]

if faltantes:
    raise FileNotFoundError(
        "Faltan estos archivos:\n" + "\n".join(faltantes)
    )

print("Panel y cuatro archivos de pronósticos encontrados.")

Panel y cuatro archivos de pronósticos encontrados.


In [4]:
panel = C.load_panel()
evaluaciones = {}
filas = []

for modelo in MODELOS:
    resultado = C.evaluate(
        C.PRED_DIR / f"{modelo}.parquet",
        panel,
    )
    evaluaciones[modelo] = resultado

    filas.append({
        "modelo": modelo,
        **resultado["global"],
        "MASE_22_28dic": resultado["xmas"]["MASE"],
        "pronosticos": resultado["n"],
    })

tabla = (
    pd.DataFrame(filas)
    .set_index("modelo")
    .sort_values("MASE")
)

display(tabla.round(4))

tabla.to_csv(C.ROOT / "results" / "metricas_recurrentes_B.csv")

print("Tabla guardada en results/metricas_recurrentes_B.csv")

,MAE,RMSE,WAPE,MAPE,MAPE_n_excluidos,MASE,MASE_22_28dic,pronosticos
modelo,,,,,,,,
seq2seq_lstm,40.5994,192.3321,0.0753,0.1146,3018,0.9585,1.6737,220800
seq2seq_gru,41.2605,199.5308,0.0765,0.1148,3018,0.9589,1.8300,220800
gru_multi,42.3790,183.4044,0.0786,0.1123,3018,1.0228,1.7958,220800
lstm_multi,43.0896,192.3069,0.0799,0.1232,3018,1.0231,1.7869,220800


Tabla guardada en results/metricas_recurrentes_B.csv
